<a href="https://colab.research.google.com/github/umtklnder/Makine-renme/blob/main/Titanic_Veri_%C4%B0%C5%9Fleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

if not (os.path.exists("train.csv") and os.path.exists("test.csv")):
  from google.colab import files

  print(
      "Lütfen Kaggle'dan indirdiğiniz train.csv ve test.csv dosyalarını seçin:"
  )
  files.upload()

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Eğitim boyutu:", train.shape, "| Test boyutu:", test.shape)
print("Eğitim verisi - ilk 5 satır:")
print(train.head().to_string(index=False))

In [ ]:
print("Eğitim eksikleri:")
print(train.isna().sum().to_string())

print("\nTest eksikleri:")
print(test.isna().sum().to_string())

print("\nHedef sınıf dağılımı:")
print(train["Survived"].value_counts().sort_index().to_string())

print("\nSayısal sütunların özeti:")
print(
    train[["Age", "Fare", "Pclass", "SibSp", "Parch"]]
    .describe()
    .round(2)
    .to_string()
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.countplot(data=train, x="Survived", ax=axes[0], color="#4682b4")
axes[0].set(
    title="Hayatta kalma sayıları",
    xlabel="Survived (0 / 1)",
    ylabel="Yolcu sayısı",
)

sns.barplot(
    data=train, x="Sex", y="Survived", ax=axes[1], color="#5a9e85", errorbar=None
)
axes[1].set(
    title="Cinsiyete göre hayatta kalma oranı",
    xlabel="Cinsiyet",
    ylabel="Oran",
    ylim=(0, 1),
)

sns.barplot(
    data=train,
    x="Pclass",
    y="Survived",
    ax=axes[2],
    color="#b47a68",
    errorbar=None,
)
axes[2].set(
    title="Sınıfa göre hayatta kalma oranı",
    xlabel="Yolcu sınıfı",
    ylabel="Oran",
    ylim=(0, 1),
)

fig.tight_layout()
plt.show()

In [ ]:
# Doldurma değerlerini sadece eğitim setinden öğreniyoruz
age_median = train["Age"].median()
embarked_mode = train["Embarked"].mode(dropna=True).iloc[0]
fare_median = train["Fare"].median()

print(
    f"Eğitimden öğrenilen değerler: Age medyanı={age_median:g}, Embarked"
    f" modu={embarked_mode}, Fare medyanı={fare_median:.4f}"
)

# Kullanacağımız temel sütunlar
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]


def preprocess(df):
  X = df[feature_cols].copy()
  # Aile büyüklüğü türetme
  X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
  X = X.drop(columns=["SibSp", "Parch"])

  # Eksik değerleri doldurma
  X["Age"] = X["Age"].fillna(age_median)
  X["Embarked"] = X["Embarked"].fillna(embarked_mode)
  X["Fare"] = X["Fare"].fillna(fare_median)

  # Kategorik değişkenleri sayısallaştırma (One-Hot Encoding)
  return pd.get_dummies(X, columns=["Sex", "Embarked"], dtype=int)


y_train = train["Survived"].copy()
X_train = preprocess(train)
X_test = preprocess(test).reindex(columns=X_train.columns, fill_value=0)

print("Ön işlenmiş eğitim:", X_train.shape, "| Test:", X_test.shape)
print("Özellikler:", X_train.columns.tolist())
print(
    "Eksik değer toplamları: eğitim=",
    int(X_train.isna().sum().sum()),
    ", test=",
    int(X_test.isna().sum().sum()),
)
print("\nİlk 5 eğitim satırı:\n" + X_train.head().to_string(index=False))

In [ ]:
corr = pd.concat([y_train.rename("Survived"), X_train], axis=1).corr(
    numeric_only=True
)

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Eğitim verisi: korelasyon matrisi")
fig.tight_layout()
plt.show()

print(
    "Genel hayatta kalma oranı: %{:.1f}".format(train["Survived"].mean() * 100)
)
print("Cinsiyete göre hayatta kalma oranı (%):")
print((train.groupby("Sex")["Survived"].mean() * 100).round(1).to_string())
print("Sınıfa göre hayatta kalma oranı (%):")
print((train.groupby("Pclass")["Survived"].mean() * 100).round(1).to_string())

In [ ]:
X_train.to_csv("X_train_preprocessed.csv", index=False)
y_train.to_frame().to_csv("y_train.csv", index=False)
X_test.to_csv("X_test_preprocessed.csv", index=False)

print("Kaydedildi: X_train_preprocessed.csv, y_train.csv, X_test_preprocessed.csv")